# 再帰的分割（Recursive Character Splitting）デモ

このノートブックでは、実用的なRAGでよく使われる **再帰的分割** の仕組みを体験します。

## 再帰的分割とは？

「段落 → 文 → 単語」の順に区切り文字の優先度を決めておき、
チャンクが目標サイズに収まるまで **階層的に分割を繰り返す** 方法です。

```
区切り文字の優先順位（日本語の場合）
  1位: \n\n  （段落）
  2位: 。    （文末）
  3位: 、    （読点）
  4位: " "   （スペース）
  5位: ""    （文字単位）

→ まず段落で分割してみる
→ まだ大きすぎる → 文末で分割してみる
→ まだ大きすぎる → 読点で分割してみる ...
```

段落分割と固定長分割の **いいとこ取り** をしたアプローチです。

## 0. ライブラリのインストール

In [ ]:
!pip install langchain langchain-text-splitters sentence-transformers \
             matplotlib scikit-learn numpy japanize-matplotlib -q
print('インストール完了！')

---
## ステップ1: サンプル文書の準備

前回と同じ機械学習の説明文を使います。
意図的に長さの異なる段落を混ぜています。

In [ ]:
document = """
機械学習は、コンピュータがデータから自動的に学習する技術です。従来のプログラミングでは、人間がルールを明示的に記述する必要がありました。しかし機械学習では、大量のデータを与えることでモデルが自らパターンを発見します。この技術は1950年代から研究が始まり、近年のデータ量増加とコンピュータ性能向上により急速に発展しました。現在ではほぼすべての産業分野で活用されており、その応用範囲は今後さらに広がると予測されています。

機械学習には主に3つの学習方法があります。教師あり学習は、正解ラベル付きのデータを使ってモデルを訓練します。教師なし学習は、ラベルなしデータからパターンやクラスタを発見します。強化学習は、試行錯誤を通じて報酬を最大化する方策を学びます。

ディープラーニングは機械学習の一分野で、多層のニューラルネットワークを使います。画像認識、音声認識、自然言語処理など幅広い分野で優れた性能を発揮しています。GPUの発展と大規模データの普及により、ディープラーニングは急速に進化しました。特に2012年のAlexNetによるImageNet大会での圧勝は、ディープラーニングブームの火付け役となりました。その後、ResNet、BERT、GPTなど革新的なアーキテクチャが次々と登場し、各分野で人間レベルを超える性能を実現しています。

自然言語処理は、コンピュータが人間の言語を理解・生成する技術です。テキスト分類、機械翻訳、質問応答システムなどに活用されています。近年はTransformerアーキテクチャが主流となり、ChatGPTのような大規模言語モデルが登場しました。
"""

print(f'文書の文字数: {len(document)}文字')
print('\n--- 文書内容 ---')
print(document)

---
## ステップ2: 3つのチャンク化方法を比較

同じ文書に対して3つの方法を適用して違いを見てみましょう。

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

CHUNK_SIZE = 120
CHUNK_OVERLAP = 20

# ① 固定長分割
fixed_splitter = CharacterTextSplitter(
    separator='',
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
fixed_chunks = fixed_splitter.split_text(document)

# ② 段落分割（\n\nのみ）
para_splitter = CharacterTextSplitter(
    separator='\n\n',
    chunk_size=CHUNK_SIZE,
    chunk_overlap=0,
    is_separator_regex=False,
)
para_chunks = para_splitter.split_text(document)

# ③ 再帰的分割
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=['\n\n', '。', '、', ' ', ''],
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
recursive_chunks = recursive_splitter.split_text(document)

print(f'チャンクサイズ上限: {CHUNK_SIZE}文字、オーバーラップ: {CHUNK_OVERLAP}文字')
print(f'\n① 固定長分割    : {len(fixed_chunks)}チャンク')
print(f'② 段落分割      : {len(para_chunks)}チャンク')
print(f'③ 再帰的分割    : {len(recursive_chunks)}チャンク')

---
## ステップ3: 各チャンクの中身を確認

チャンクの境界がどこで切れているかを比較します。

In [ ]:
def print_chunks(title, chunks):
    print(f'\n{'='*60}')
    print(f'{title}  （{len(chunks)}チャンク）')
    print('='*60)
    for i, chunk in enumerate(chunks):
        print(f'[{i+1}] {len(chunk)}文字 | {chunk}')
        print('-'*60)

print_chunks('① 固定長分割', fixed_chunks)
print_chunks('② 段落分割', para_chunks)
print_chunks('③ 再帰的分割', recursive_chunks)

---
## ステップ4: 再帰的分割の「再帰」を可視化

どの区切り文字が実際に使われたかを確認します。
チャンクの末尾が何で終わっているかを集計します。

In [ ]:
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
import numpy as np

def classify_split_reason(chunks, original_text, chunk_size):
    """各チャンクがどの理由で分割されたかを判定"""
    reasons = []
    for chunk in chunks:
        if len(chunk) < chunk_size * 0.7:
            # 小さいチャンク → 意味の区切りで自然に終わった
            if chunk.rstrip().endswith('。'):
                reasons.append('文末（。）で分割')
            elif chunk.rstrip().endswith('、'):
                reasons.append('読点（、）で分割')
            else:
                reasons.append('段落（\\n\\n）で分割')
        else:
            # 大きいチャンク → サイズ上限に近い
            if chunk.rstrip().endswith('。'):
                reasons.append('文末（。）で分割')
            elif chunk.rstrip().endswith('、'):
                reasons.append('読点（、）で分割')
            else:
                reasons.append('文字単位で分割')
    return reasons

reasons = classify_split_reason(recursive_chunks, document, CHUNK_SIZE)

# 集計
from collections import Counter
reason_counts = Counter(reasons)

# 棒グラフ
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左: 分割理由の内訳
labels = list(reason_counts.keys())
counts = list(reason_counts.values())
colors = ['#4A90D9', '#E67E22', '#2ECC71', '#E74C3C']
bars = axes[0].bar(labels, counts, color=colors[:len(labels)], edgecolor='white', linewidth=1.5)
axes[0].set_title('再帰的分割: どの区切り文字が使われたか', fontsize=12)
axes[0].set_ylabel('チャンク数')
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 str(count), ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_facecolor('#F8F9FA')
axes[0].tick_params(axis='x', labelsize=9)

# 右: 3手法のチャンクサイズ分布
method_chunks = {
    '①固定長': fixed_chunks,
    '②段落': para_chunks,
    '③再帰的': recursive_chunks,
}
data = [list(map(len, v)) for v in method_chunks.values()]
bp = axes[1].boxplot(data, labels=list(method_chunks.keys()),
                     patch_artist=True, widths=0.5)
box_colors = ['#4A90D9', '#E67E22', '#2ECC71']
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].axhline(y=CHUNK_SIZE, color='red', linestyle='--', alpha=0.6, label=f'上限 {CHUNK_SIZE}文字')
axes[1].set_title('3手法のチャンクサイズ分布', fontsize=12)
axes[1].set_ylabel('チャンクの文字数')
axes[1].legend()
axes[1].set_facecolor('#F8F9FA')

plt.tight_layout()
plt.savefig('recursive_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('左グラフ: 再帰的分割で実際に使われた区切り文字の内訳')
print('右グラフ: 各手法のチャンクサイズのばらつき（箱ひげ図）')

---
## ステップ5: chunk_size を変えて挙動を観察

chunk_size が小さいほど細かく切れ、大きいほど意味のまとまりを優先します。
**小・中・大** の3パターンで比較してみましょう。

In [ ]:
configs = [
    {'chunk_size':  60, 'chunk_overlap': 10, 'label': '小 (60文字)'},
    {'chunk_size': 120, 'chunk_overlap': 20, 'label': '中 (120文字)'},
    {'chunk_size': 250, 'chunk_overlap': 30, 'label': '大 (250文字)'},
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)

for ax, cfg in zip(axes, configs):
    splitter = RecursiveCharacterTextSplitter(
        separators=['\n\n', '。', '、', ' ', ''],
        chunk_size=cfg['chunk_size'],
        chunk_overlap=cfg['chunk_overlap'],
    )
    chunks = splitter.split_text(document)
    sizes = list(map(len, chunks))

    x = range(len(chunks))
    bars = ax.bar(x, sizes, color='#4A90D9', alpha=0.8, edgecolor='white')
    ax.axhline(y=cfg['chunk_size'], color='red', linestyle='--', alpha=0.7, label=f'上限')
    ax.set_title(f"{cfg['label']}\n→ {len(chunks)}チャンク", fontsize=11)
    ax.set_xlabel('チャンク番号')
    ax.set_ylabel('文字数')
    ax.set_xticks(x)
    ax.set_xticklabels([str(i+1) for i in x])
    ax.legend(fontsize=9)
    ax.set_facecolor('#F8F9FA')
    for bar, size in zip(bars, sizes):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(size), ha='center', va='bottom', fontsize=8)

plt.suptitle('chunk_size によるチャンク数・サイズの変化', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('chunk_size_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('chunk_size が大きいほどチャンク数が少なく、1チャンクに多くの情報が入ります。')
print('RAGでは検索精度とコンテキスト量のトレードオフになります。')

---
## ステップ6: ベクトル化 → 類似度検索

前回同様、再帰的分割したチャンクをベクトル化して検索します。
チャンクの質が検索精度にどう影響するかを確認しましょう。

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print('モデルをロード中...')
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print('ロード完了!')

# chunk_size=120 の再帰的チャンクでベクトル化
splitter = RecursiveCharacterTextSplitter(
    separators=['\n\n', '。', '、', ' ', ''],
    chunk_size=120,
    chunk_overlap=20,
)
chunks = splitter.split_text(document)
embeddings = model.encode(chunks)

print(f'\nチャンク数: {len(chunks)}')
print(f'ベクトル次元数: {embeddings.shape[1]}')

In [ ]:
def search(query, chunks, embeddings, top_k=3):
    query_vec = model.encode([query])
    sims = cosine_similarity(query_vec, embeddings)[0]
    ranked = sorted(enumerate(sims), key=lambda x: x[1], reverse=True)
    print(f"クエリ: '{query}'")
    print('─' * 60)
    for rank, (idx, score) in enumerate(ranked[:top_k], 1):
        print(f'【第{rank}位】 類似度: {score:.4f}')
        print(f'{chunks[idx]}')
        print()

print('=' * 60)
search('ニューラルネットワークについて教えて', chunks, embeddings)
print('=' * 60)
search('ChatGPTはどんな技術を使っている？', chunks, embeddings)
print('=' * 60)
search('機械学習の学習方法の種類は？', chunks, embeddings)

---
## ステップ7: ベクトル空間の可視化

再帰的分割のチャンクとクエリをPCAで2次元に圧縮して可視化します。

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.patches as mpatches

queries = ['ニューラルネットワーク', 'ChatGPT', '学習方法の種類']
query_vecs = model.encode(queries)

all_vecs = np.vstack([embeddings, query_vecs])
pca = PCA(n_components=2)
reduced = pca.fit_transform(all_vecs)

fig, ax = plt.subplots(figsize=(11, 7))
ax.set_facecolor('#F8F9FA')

# チャンクをプロット（色でインデックスを表現）
cmap = plt.cm.Blues
n = len(chunks)
for i in range(n):
    x, y = reduced[i]
    color = cmap(0.4 + 0.5 * i / max(n-1, 1))
    ax.scatter(x, y, c=[color], s=220, marker='o', zorder=3, edgecolors='white', linewidths=1.5)
    ax.annotate(f'C{i+1}', (x, y),
                textcoords='offset points', xytext=(7, 7),
                fontsize=9, color='#2C3E50',
                bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                          edgecolor='#BDC3C7', alpha=0.85))

# クエリをプロット
q_colors = ['#E74C3C', '#E67E22', '#9B59B6']
for i, (query, color) in enumerate(zip(queries, q_colors)):
    x, y = reduced[n + i]
    ax.scatter(x, y, c=[color], s=350, marker='*', zorder=4, edgecolors='white', linewidths=1.5)
    ax.annotate(query, (x, y),
                textcoords='offset points', xytext=(8, 8),
                fontsize=10, color=color, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                          edgecolor=color, alpha=0.9))

# チャンクの内容をサイドに表示
chunk_info = '\n'.join([f'C{i+1}: {c[:25]}...' for i, c in enumerate(chunks)])
ax.text(1.02, 0.99, chunk_info, transform=ax.transAxes,
        fontsize=7.5, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='#EBF5FB', edgecolor='#AED6F1'))

patch_chunk = mpatches.Patch(color='steelblue', label='チャンク（文書断片）')
patch_q = mpatches.Patch(color='#E74C3C', label='クエリ（検索語）')
ax.legend(handles=[patch_chunk, patch_q], loc='lower left')

ax.set_title('再帰的分割チャンクのベクトル空間（PCAで2次元圧縮）\n近い点 = 意味が似ている', fontsize=13, pad=12)
ax.set_xlabel('第1主成分')
ax.set_ylabel('第2主成分')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('recursive_vector_space.png', dpi=150, bbox_inches='tight')
plt.show()

---
## ステップ8: オーバーラップの効果を確認

オーバーラップとは、隣接するチャンク間で文字を重複させることで
**チャンク境界で文脈が途切れる問題** を緩和する仕組みです。

In [ ]:
splitter_no_overlap = RecursiveCharacterTextSplitter(
    separators=['\n\n', '。', '、', ' ', ''],
    chunk_size=120, chunk_overlap=0,
)
splitter_with_overlap = RecursiveCharacterTextSplitter(
    separators=['\n\n', '。', '、', ' ', ''],
    chunk_size=120, chunk_overlap=30,
)

chunks_no = splitter_no_overlap.split_text(document)
chunks_ov = splitter_with_overlap.split_text(document)

print('=== オーバーラップなし ===')
for i, c in enumerate(chunks_no[:4]):
    print(f'[{i+1}] {c}')
    print()

print('\n=== オーバーラップあり（30文字） ===')
for i, c in enumerate(chunks_ov[:4]):
    print(f'[{i+1}] {c}')
    print()

print('→ オーバーラップありでは隣のチャンクの末尾が次のチャンクの先頭に含まれます。')
print('  境界付近の情報を両方のチャンクが保持できるため、検索の取りこぼしを減らせます。')

---
## まとめ

| 手法 | メリット | デメリット |
|------|----------|------------|
| 固定長分割 | シンプル・高速 | 文の途中で切れる |
| 段落分割 | 意味のまとまりを保つ | チャンクサイズがバラバラ |
| **再帰的分割** | **サイズを保ちながら意味も考慮** | パラメータ調整が必要 |

### 再帰的分割のポイント
- `separators` は言語に合わせて設定（日本語は `。` `、` を含める）
- `chunk_size` が小さい → 精密な検索、コンテキストが少ない
- `chunk_size` が大きい → コンテキストが豊富、ノイズが増える
- `chunk_overlap` を設定してチャンク境界での情報損失を緩和する